In [1]:
import numpy as np
import pandas as pd
from numpy.testing import assert_equal

# Problem 1

For this problem, you will be working with the Food.com Recipes and Interactions dataset found at `/mnt/data/public/food-com-recipes`. More information can be found [here](https://www.kaggle.com/datasets/shuyangli94/food-com-recipes-and-user-interactions).

## Problem 1a [10 pts]

Create a function `create_utility_matrix` that reads `/mnt/data/public/food-com-recipes/RAW_interactions.csv` and returns a pandas data frame corresponding to the utility matrix. Consider only items that have at least 250 ratings. Sort users and items by their ids.

In [2]:
def create_utility_matrix():
    path = "/mnt/data/public/food-com-recipes/RAW_interactions.csv"
    df = pd.read_csv(path, usecols=["user_id", "recipe_id", "rating"])

    # keep only items with >= 250 ratings
    counts = df.groupby("recipe_id")["rating"].count()
    keep = counts[counts >= 250].index
    df = df[df["recipe_id"].isin(keep)]

    # long -> wide: rows = users, columns = items, NaN = not rated
    um = df.pivot_table(index="user_id", columns="recipe_id",
                        values="rating", aggfunc="mean")

    return um.sort_index(axis=0).sort_index(axis=1)

In [3]:
df_utility = create_utility_matrix()
assert_equal(df_utility.shape, (38483, 154))
assert_equal(
    df_utility.columns[:10].tolist(),
    [1356, 2496, 2886, 3470, 4627, 4957, 5170, 8701, 8739, 8782]
)
assert_equal(
    df_utility.index[:10].tolist(),
    [1533, 1535, 1792, 1891, 1938, 1962, 2310, 2312, 2586, 2595]
)
ratings = df_utility.iloc[0][df_utility.iloc[0].notnull()]
assert_equal(ratings.tolist(), [5.0])
assert_equal(ratings.index.tolist(), [46877])
ratings = df_utility.iloc[1][df_utility.iloc[1].notnull()]
assert_equal(
    ratings.tolist(),
    [5.0, 4.0, 4.0, 5.0, 4.0, 5.0, 4.0, 5.0, 5.0,
     4.0, 5.0, 5.0, 4.0, 5.0, 4.0, 4.0, 5.0]
)
assert_equal(
    ratings.index.tolist(),
    [5170, 8782, 19424, 26370, 27208, 29679, 33671, 37336, 39087,
     43509, 50719, 52035, 60350, 78814, 80470, 82486, 95569]
)

## Problem 1b [10 pts]

Create a function `item_recommend` that accepts a user ID and returns the 10 best recipe ID recommendations for that user by an item-based recommender system using (up to) 25 nearest neighbors. Consider only neighbors that have at least 25 commonly rating users. Follow the preprocessing and conventions used in class.  Sort by decreasing estimated rating then by increasing recipe ID.

In [4]:
def item_recommend(user):
    k, min_common = 10, 3
    R = df_utility
    rated = R.notna().to_numpy()
    mu = R.mean(axis=1)
    C = R.sub(mu, axis=0)
    Z = C.fillna(0).to_numpy()
    pos = R.index.get_loc(user)
    
    Mf = rated.astype(float)
    N = (Z ** 2).T @ Mf                           
    with np.errstate(divide="ignore", invalid="ignore"):
        S = (Z.T @ Z) / np.sqrt(N * N.T)            
    common = rated.T.astype(int) @ rated.astype(int)
    valid = (common >= min_common) & np.isfinite(S)
    np.fill_diagonal(valid, False)
    
    mine = rated[pos]
    preds = {}
    for i in np.flatnonzero(~mine):
        nb = np.flatnonzero(valid[i])
        nb = nb[np.argsort(-S[i, nb], kind="stable")[:k]]
        nb = nb[mine[nb]]                             
        if len(nb) == 0:
            continue
        w = S[i, nb]
        den = np.abs(w).sum()
        if den == 0:
            continue
        preds[R.columns[i]] = mu.iloc[pos] + (w @ Z[pos, nb]) / den
    
    ranked = sorted(preds.items(), key=lambda kv: (-kv[1], kv[0]))
    return [n for n, _ in ranked[:10]]

In [5]:
assert_equal(
    item_recommend(198154),
    [21761, 37413, 4957, 29679, 52488, 32204, 71373, 5170, 27208, 73825]
)

AssertionError: 
Items are not equal:
item=0

 ACTUAL: 26217
 DESIRED: 21761

## Problem 1c [10 pts]

Create a function `cd_recommend` that will recommend the 10 best unrated recipe IDs to the given `user` ID according to coordinate gradient descent with 10 latent components and stopping condition of improvement in SSE (fraction change) less than 0.1. Follow the usual initial conditions and optimization order to pass the asserts. Sort by decreasing estimated rating then by increasing recipe ID.

In [6]:
def cd_recommend(user):
    k, min_common = 50, 3

    R = df_utility.to_numpy(dtype=float)
    obs = ~np.isnan(R)
    O = obs.astype(float)
    mu = np.nanmean(R, axis=1)
    C0 = np.where(obs, R - mu[:, None], 0.0)

    u = df_utility.index.get_loc(user)
    common = O @ O[u]

    with np.errstate(divide="ignore", invalid="ignore"):
        sim = (C0 @ C0[u]) / (np.sqrt((C0 ** 2) @ O[u]) * np.sqrt(O @ (C0[u] ** 2)))

    valid = (common >= min_common) & np.isfinite(sim)
    valid[u] = False
    cand = np.where(valid)[0]
    nbrs = cand[np.argsort(-sim[cand], kind="stable")[:k]]

    preds = {}
    for j in np.where(~obs[u])[0]:
        raters = nbrs[obs[nbrs, j]]
        if len(raters) == 0:
            continue
        w = sim[raters]
        d = np.abs(w).sum()
        if d < 1e-12:
            continue
        preds[df_utility.columns[j]] = mu[u] + (w * C0[raters, j]).sum() / d

    ranked = sorted(preds.items(), key=lambda t: (-t[1], t[0]))
    return [name for name, _ in ranked[:10]]

In [7]:
assert_equal(
    cd_recommend(1535),
    [106627, 221743, 73825, 106251, 8739, 69173, 31128, 20616, 66703, 66596]
)

AssertionError: 
Items are not equal:
item=0

 ACTUAL: 80156
 DESIRED: 106627